# Named Entity Recognition (NER) on News Titles using Hugging Face

In this experiment, a pretrained BERT-based Named Entity Recognition model from Hugging Face is used to identify named entities from the `Title` column of the news dataset.

The extracted entities are stored in a new `NER` column as dictionaries, where:

- Key = Entity
- Value = Entity Type

Example:

{
    "Elon Musk": "PERSON",
    "Tesla": "ORGANIZATION",
    "Germany": "PLACE"
}

In [1]:
!pip install -q transformers torch

## 1. Import Required Libraries

Pandas is used for dataset handling, PyTorch is used to detect GPU availability, and the Hugging Face pipeline is used for Named Entity Recognition.

In [2]:
import pandas as pd
import torch

from transformers import pipeline

## 2. Load the News Dataset

The dataset contains news articles and their titles. NER will be applied only to the `Title` column.

In [4]:
FILE = "./sample_data/corrected_news.csv"

df = pd.read_csv(FILE)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset Shape: (4997, 10)

Columns:
['URL', 'Title', 'Category', 'Group Leader', 'Original_Category', 'Title_Normalized', 'Original_Row_ID', 'Original_Majority', 'Final_Category', 'Label_Source']


,URL,Title,Category,Group Leader,Original_Category,Title_Normalized,Original_Row_ID,Original_Majority,Final_Category,Label_Source
0,https://asiatimes.com/2025/10/vietnam-airlines...,Vietnam Airlines data leak exposes a crisis of...,Technology,NaN,Technology,vietnam airlines data leak exposes a crisis of...,0,Technology,Technology,TITLE
1,https://asiatimes.com/2025/10/doj-seizes-15-bi...,DOJ seizes $15 billion bitcoin in SE Asia cryp...,Markets,NaN,Markets,doj seizes $15 billion bitcoin in se asia cryp...,1,Markets,Markets,TITLE
2,https://www.fool.com/investing/2025/10/15/too-...,Think It's Too Late to Buy IonQ? Here's the 1 ...,Markets,NaN,Markets,think it's too late to buy ionq? here's the 1 ...,2,Markets,Markets,URL
3,https://www.fool.com/investing/2025/10/15/inte...,Intel's Crucial Panther Lake Chips Start Produ...,Technology,NaN,Technology,intel's crucial panther lake chips start produ...,3,Technology,Technology,TITLE
4,https://www.fool.com/investing/2025/10/15/3-ro...,3 Robotics Stocks to Buy in October,Markets,NaN,Markets,3 robotics stocks to buy in october,4,Markets,Markets,TITLE+URL


## 3. Load the Pretrained Hugging Face NER Model

The `dslim/bert-base-NER` model is a BERT-based model fine-tuned for Named Entity Recognition.

It recognizes four main entity types:

- PER → Person
- ORG → Organization
- LOC → Location
- MISC → Miscellaneous

The aggregation strategy combines related subword tokens into entity spans.

In [5]:
device = 0 if torch.cuda.is_available() else -1

ner = pipeline(
    task="token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=device
)

print("NER model loaded successfully.")

if torch.cuda.is_available():
    print("Running on GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

NER model loaded successfully.
Running on GPU: Tesla T4


## 4. Test the NER Model on One News Title

Before applying the model to the complete dataset, it is tested on one title to inspect its predictions.

In [6]:
sample_title = df["Title"].iloc[0]

print("Title:")
print(sample_title)

print("\nNER Output:")
ner(sample_title)

Title:
Vietnam Airlines data leak exposes a crisis of transparency

NER Output:


[{'entity_group': 'ORG',
  'score': np.float32(0.9966762),
  'word': 'Vietnam Airlines',
  'start': 0,
  'end': 16}]

## 5. Define Entity Label Mapping

The original Hugging Face labels are converted into more readable names.

For example:

PER → PERSON  
ORG → ORGANIZATION  
LOC → PLACE  
MISC → MISCELLANEOUS

In [7]:
LABEL_MAP = {
    "PER": "PERSON",
    "ORG": "ORGANIZATION",
    "LOC": "PLACE",
    "MISC": "MISCELLANEOUS"
}

## 6. Create a Function to Convert NER Output into Dictionary Format

For every news title, the detected entities are converted into a dictionary.

The entity text becomes the dictionary key and its entity type becomes the value.

Example:

{
    "Elon Musk": "PERSON",
    "Tesla": "ORGANIZATION",
    "Germany": "PLACE"
}

In [8]:
def convert_to_ner_dict(entities):

    ner_dict = {}

    for entity in entities:

        entity_text = entity["word"].strip()
        entity_label = entity["entity_group"]

        if entity_text:
            ner_dict[entity_text] = LABEL_MAP.get(
                entity_label,
                entity_label
            )

    return ner_dict

## 7. Apply NER to the Complete Title Column

The pretrained BERT NER model is now applied to every news title.

The predictions are then converted into dictionary format and stored in a new column named `NER`.

In [9]:
titles = df["Title"].fillna("").astype(str).tolist()

ner_results = ner(
    titles,
    batch_size=32
)

df["NER"] = [
    convert_to_ner_dict(result)
    for result in ner_results
]

## 8. View the Extracted Named Entities

The `Title` column contains the original news title, while the `NER` column contains the entities detected by the Hugging Face model.

In [10]:
df[["Title", "NER"]].head(10)

,Title,NER
0,Vietnam Airlines data leak exposes a crisis of...,{'Vietnam Airlines': 'ORGANIZATION'}
1,DOJ seizes $15 billion bitcoin in SE Asia cryp...,"{'DOJ': 'ORGANIZATION', 'SE Asia': 'PLACE'}"
2,Think It's Too Late to Buy IonQ? Here's the 1 ...,{'Ion': 'MISCELLANEOUS'}
3,Intel's Crucial Panther Lake Chips Start Produ...,"{'Intel': 'ORGANIZATION', 'Cru': 'MISCELLANEOU..."
4,3 Robotics Stocks to Buy in October,{}
5,Think It's Too Late to Buy Applied Digital? He...,{'Applied Digital': 'ORGANIZATION'}
6,Transcript: UK Budget blues,{'UK Budget': 'MISCELLANEOUS'}
7,Is Investing in Nebius Group Stock a Once-in-a...,"{'Nebius Group': 'ORGANIZATION', 'Lifetime': '..."
8,What Are 3 Great Tech Stocks to Buy Right Now?,{}
9,"Down 27% From Its High, Is CoreWeave a Top AI ...",{'CoreWea': 'ORGANIZATION'}


## 9. Inspect Individual NER Predictions

A few news titles and their extracted entities are printed to make the results easier to inspect.

In [11]:
for i in range(10):

    print("Title:")
    print(df.loc[i, "Title"])

    print("NER:")
    print(df.loc[i, "NER"])

    print("-" * 80)

Title:
Vietnam Airlines data leak exposes a crisis of transparency
NER:
{'Vietnam Airlines': 'ORGANIZATION'}
--------------------------------------------------------------------------------
Title:
DOJ seizes $15 billion bitcoin in SE Asia crypto scam bust
NER:
{'DOJ': 'ORGANIZATION', 'SE Asia': 'PLACE'}
--------------------------------------------------------------------------------
Title:
Think It's Too Late to Buy IonQ? Here's the 1 Reason Why There's Still Time
NER:
{'Ion': 'MISCELLANEOUS'}
--------------------------------------------------------------------------------
Title:
Intel's Crucial Panther Lake Chips Start Production
NER:
{'Intel': 'ORGANIZATION', 'Cru': 'MISCELLANEOUS', '##cial Panther Lake': 'MISCELLANEOUS', 'Chips Start': 'ORGANIZATION'}
--------------------------------------------------------------------------------
Title:
3 Robotics Stocks to Buy in October
NER:
{}
--------------------------------------------------------------------------------
Title:
Think It's Too 

## 10. Save the Final Dataset

The final dataset containing the extracted Named Entities is saved as a new CSV file.

In [13]:
OUTPUT_FILE = "news_with_ner.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Dataset saved successfully as:", OUTPUT_FILE)

Dataset saved successfully as: news_with_ner.csv
